In [14]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# Theme of the Project

This project investigates the mathematical foundations of cryptography through the implementation and analysis of classical and modern encryption schemes. Using Python and NumPy, we explore modular arithmetic and linear algebra as the theoretical basis for the Caesar cipher, Vigenère cipher, and the matrix-based Hill cipher, followed by a simplified demonstration of RSA encryption and cryptographic hash functions. Each notebook focuses on the underlying mathematics of one cipher, bridging the gap between abstract mathematical concepts and their practical application in secure communication.

## Table of Contents

1. Mathematical Foundations
   - 1.1 Modular Arithmetic
   - 1.2 GCD & The Euclidean Algorithm
   - 1.3 Modular Inverse
   - 1.4 Matrix Operations in Cryptography

2. Classical Ciphers
   - 2.1 Caesar Cipher
   - 2.2 Vigenère Cipher

3. Hill Cipher
   - 3.1 Encryption
   - 3.2 Decryption

4. Modern Cryptography
   - 4.1 RSA
   - 4.2 Hashing
   - 4.3 AES — Symmetric Encryption
   - 4.4 Cryptographically Secure Randomness
   - 4.5 The Quantum Computing Threat

5. Conclusion

## 1. Mathematical Foundations

Before implementing any cipher, we establish the three mathematical 
tools that all cryptographic schemes in this project rely on:

- **Modular arithmetic** — the algebra of remainders, forming the 
  basis of all letter transformations
- **Greatest Common Divisor (GCD)** — used to determine whether a 
  number or matrix is invertible in modular space
- **Matrix operations (mod n)** — the foundation of block ciphers 
  such as the Hill cipher, where encryption is a linear transformation
  over a finite field

### Modular Arithmetic

Modular arithmetic is a mathematical system where integers "wrap around" 
after exceeding a given value called the **modulus**. Formally:

$$a \equiv b \pmod{m} \iff m \mid (a - b)$$

We say $a$ is congruent to $b$ modulo $m$ if $m$ divides their difference.

For example:

$$3 \equiv 15 \pmod{12} \quad \text{since } 12 \mid (15 - 3)$$
$$70 \equiv 10 \pmod{60} \quad \text{since } 60 \mid (70 - 10)$$

In cryptography, modular arithmetic keeps all values within a fixed range.
For classical ciphers this range is $\mathbb{Z}_{26} = \{0, 1, \dots, 25\}$,
where each integer represents a letter of the English alphabet:

$$27 \equiv 1 \pmod{26}$$
$$34 \equiv 8 \pmod{26}$$

The following cell shows the implementation in python:

In [15]:
print(27 % 26)  # 1
print(34 % 26)  # 8
print(-6 % 26)

1
8
20



### Greatest Common Divisor (GCD)

The Greatest Common Divisor of two integers is the largest integer 
that divides both without a remainder:

$$\gcd(a, b) = \max\{d \in \mathbb{Z}^+ : d \mid a \text{ and } d \mid b\}$$

Two integers are called **coprime** if their GCD equals 1:

$$\gcd(a, b) = 1$$

This is not an edge case — it is the fundamental requirement in 
cryptography. A number $a$ has a modular inverse mod $m$ **if and 
only if** $\gcd(a, m) = 1$.

### The Euclidean Algorithm

Rather than factoring both numbers, we use the recursive observation:

$$\gcd(a, b) = \gcd(b,\ a \bmod b)$$

repeated until the remainder reaches zero. For example:

$$\gcd(48, 18) \rightarrow \gcd(18, 12) \rightarrow \gcd(12, 6) 
\rightarrow \gcd(6, 0) = 6$$

In the context of this project, the condition we require is:

$$\gcd(a, 26) = 1$$

For example $\gcd(3, 26) = 1$ so $3$ is a valid key, 
while $\gcd(13, 26) = 13$ so $13$ is not.

The following cell shows the implementation in python:

In [16]:
def findGCD(a,b):
    if b == 0:
        return a
    return findGCD(b,a % b)    

In [17]:
print(findGCD(3,26))
print(findGCD(18,48))
print(findGCD(2,4))


1
6
2


Sumpy has a built in function for finding GCD will we will use for now on for simplicity reasons but first lets check if our method is conssitent with the sumpy method.

In [18]:
assert sp.gcd(3,26) == findGCD(3,26)
assert sp.gcd(18,48) == findGCD(18,48)
assert sp.gcd(2,4) == findGCD(2,4)
assert sp.gcd(47,123) == findGCD(47,123) 


### Modular Inverse

The modular inverse of $a$ modulo $m$ is an integer $x$ such that:

$$a \cdot x \equiv 1 \pmod{m}$$

For example, the modular inverse of $3$ mod $26$ is $9$, since:

$$3 \times 9 = 27 \equiv 1 \pmod{26}$$

A modular inverse exists **if and only if**:

$$\gcd(a, m) = 1$$

We find it using the **Extended Euclidean Algorithm**, which finds 
integers $x$ and $y$ satisfying Bézout's identity:

$$ax + my = \gcd(a, m)$$

When $\gcd(a, m) = 1$, the coefficient $x$ reduced mod $m$ is 
the modular inverse of $a$.

The following cell shows the implementation in python: 

In [19]:
def extended_euclidian(a,modular):
    curr_number = a
    next_number = modular
    current_coefficient = 1
    next_coefficient = 0
    while next_number != 0 : 
        q = curr_number // next_number
        curr_number , next_number = next_number, curr_number - next_number * q
        current_coefficient , next_coefficient = next_coefficient , current_coefficient - next_coefficient * q
    return curr_number,current_coefficient 
def mod_inverse(a,modular):
    gcd , x = extended_euclidian(a,modular)
    if gcd != 1:
        return None
    return x % modular         

In [20]:
print(mod_inverse(3 , 26))

9


### Matrix Operations

Building upon the fundamental matrix operations of addition, multiplication 
and linear transformations, we introduce three additional properties 
required for the Hill cipher.

**Determinant**

The determinant is a scalar value that captures whether a matrix is 
invertible. For a $2 \times 2$ matrix:

$$K = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$$

it is computed as:

$$\det(K) = ad - bc$$

This is a special case of the general $n \times n$ cofactor expansion 
along row $i$:

$$\det(A) = \sum_{j=1}^{n} (-1)^{i+j} \, a_{ij} \, M_{ij}$$

where $M_{ij}$ is the determinant of the submatrix formed by deleting 
row $i$ and column $j$, and $(-1)^{i+j}$ is the sign of the cofactor. 
Throughout this project all key matrices are $2 \times 2$, so the 
formula $ad - bc$ is sufficient.

Geometrically, the determinant represents the signed area of the 
parallelogram formed by the two column vectors of $K$. When 
$\det(K) = 0$, the two columns are linearly dependent — the matrix 
is singular and has no inverse.

Over $\mathbb{R}$ this is the only condition for non-invertibility. 
Over $\mathbb{Z}_{26}$ the condition is stricter — the determinant 
must additionally satisfy:

$$\gcd(\det(K),\ 26) = 1$$

For example, given:

$$K = \begin{pmatrix} 3 & 3 \\ 2 & 5 \end{pmatrix}$$

$$\det(K) = 3 \cdot 5 - 3 \cdot 2 = 9, \qquad \gcd(9, 26) = 1 \checkmark$$

In [21]:
def determinant_for_2n_matrix(matrix) -> int:
    return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]

In [22]:
matrix = np.array([[3, 3],
              [2, 5]])
print(determinant_for_2n_matrix(matrix))
print(sp.gcd(determinant_for_2n_matrix(matrix),26))

9
1


Sumpy also has a built in function for getting the determinant whcich we will use from now on but first lets check if our method is consitent with the sumpy method.

In [23]:
assert determinant_for_2n_matrix(matrix) == sp.Matrix(matrix).det()

**Adjugate**

The adjugate is the matrix that satisfies:

$$K \cdot \text{adj}(K) = \det(K) \cdot I$$

Dividing both sides by $\det(K)$ yields the inverse — this is the 
algebraic purpose of the adjugate. It is defined as the transpose 
of the cofactor matrix. For an $n \times n$ matrix, each entry is:

$$\text{adj}(A)_{ij} = (-1)^{i+j} \, M_{ji}$$

where $M_{ji}$ is the determinant of the submatrix formed by 
deleting row $j$ and column $i$, and $(-1)^{i+j}$ is the 
alternating sign pattern:

$$\begin{pmatrix} + & - & + & \cdots \\ - & + & - & \cdots \\ 
+ & - & + & \cdots \\ \vdots & & & \ddots \end{pmatrix}$$

For a $2 \times 2$ matrix this reduces to a direct formula — 
swap the diagonal elements and negate the off-diagonal elements:

$$\text{adj}\begin{pmatrix} a & b \\ c & d \end{pmatrix} = 
\begin{pmatrix} d & -b \\ -c & a \end{pmatrix}$$

The fundamental algebraic property of the adjugate is:

$$K \cdot \text{adj}(K) = \det(K) \cdot I$$

Applying this to our example:

$$\text{adj}(K) = \begin{pmatrix} 5 & -3 \\ -2 & 3 \end{pmatrix}$$

In [24]:
def adjugate_for_2n_matrix(matrix) -> np.array:
    return np.array([[matrix[1][1] , -matrix[0][1]],
                     [-matrix[1][0],matrix[0][0]]])

In [25]:
adj = adjugate_for_2n_matrix(matrix)

print(adj)
assert np.array_equal(matrix @ adj, sp.Matrix(matrix).det() * np.eye(2, dtype=int))
#assert matrix @ adj ==  sp.Matrix(matrix).det() * np.array([[1,0],[0,1]]).all()

[[ 5 -3]
 [-2  3]]



**Modular Matrix Inverse**

Over $\mathbb{R}$ the matrix inverse is given by:

$$K^{-1} = \frac{1}{\det(K)} \cdot \text{adj}(K)$$

Over $\mathbb{Z}_{26}$, scalar division does not exist — it is 
replaced by the modular inverse of the determinant, established 
in Section 1.3:

$$K^{-1} \equiv \det(K)^{-1} \cdot \text{adj}(K) \pmod{26}$$

This formula is valid if and only if $\gcd(\det(K),\ 26) = 1$. 
The computation proceeds in three steps.

**Step 1** — verify the invertibility condition:

$$\det(K) = ad - bc, \qquad \gcd(\det(K),\ 26) = 1$$

**Step 2** — find the modular inverse of the determinant:

$$\det(K)^{-1} \equiv x \pmod{26} \quad \text{where} \quad 
x \cdot \det(K) \equiv 1 \pmod{26}$$

**Step 3** — multiply and reduce every entry modulo 26:

$$K^{-1} \equiv \det(K)^{-1} \cdot \begin{pmatrix} d & -b \\ -c & a 
\end{pmatrix} \pmod{26}$$

Note that negative entries arising from the adjugate are reduced 
into $\mathbb{Z}_{26}$ by adding 26:

$$-9 \bmod 26 = 17, \qquad -6 \bmod 26 = 20$$

Applying all three steps to our example:

$$9^{-1} \equiv 3 \pmod{26} \quad \text{since} \quad 
9 \times 3 = 27 \equiv 1 \pmod{26}$$

$$K^{-1} \equiv 3 \cdot \begin{pmatrix} 5 & -3 \\ -2 & 3 \end{pmatrix} 
\equiv \begin{pmatrix} 15 & -9 \\ -6 & 9 \end{pmatrix} 
\equiv \begin{pmatrix} 15 & 17 \\ 20 & 9 \end{pmatrix} \pmod{26}$$

The result satisfies the identity:

$$K \cdot K^{-1} \equiv \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} 
\pmod{26}$$

A $2 \times 2$ integer matrix $K$ is a valid Hill cipher key 
**if and only if**:

$$\gcd(\det(K),\ 26) = 1$$

This ensures $K^{-1} \pmod{26}$ exists and that decryption is 
uniquely defined.

The following implements these three operations step by step, 
then verifies the identity $K \cdot K^{-1} \equiv I \pmod{26}$ 
both manually and using SymPy:

In [27]:
def matrix_modular_inverse(matrix):
    det = sp.Matrix(matrix).det()
    det_mod_inverse = mod_inverse(det,26)
    adj = adjugate_for_2n_matrix(matrix)
    result = (det_mod_inverse * adj) % 26
    return result
matrix_inversed = matrix_modular_inverse(matrix)
sympy_inversed = sp.Matrix(matrix).inv_mod(26)

assert np.array_equal(matrix_inversed ,sympy_inversed)
assert np.array_equal((matrix_inversed @ matrix) % 26,  np.eye(2,dtype = int))
assert np.array_equal((sympy_inversed @ matrix) % 26,  np.eye(2,dtype = int))